# PlantVillageVQA — MiniCPM5-1B + frozen SigLIP, gated cross-attention
Thin Kaggle driver. Add the **PlantVillageVQA** dataset as a Notebook input, enable **GPU T4 x2**, then run top-to-bottom.
Clone/upload the project so `configs/`, `data/`, `models/`, `train.py`, `eval.py` are in the working dir.

In [ ]:
# --- environment ---
# torch is preinstalled (CUDA build). NO flash-attn.
!pip -q install -U transformers datasets accelerate sentencepiece pillow pyyaml matplotlib
import torch
print('cuda', torch.cuda.is_available(), 'devices', torch.cuda.device_count())
print('capability', torch.cuda.get_device_capability(0), '(7,5) => T4/fp16')

In [ ]:
# --- point config at the Kaggle-mounted dataset ---
import yaml, glob, os
cfg_path = 'configs/default.yaml'
cfg = yaml.safe_load(open(cfg_path))
# find the mounted dataset root (folder containing the CSV)
cands = glob.glob('/kaggle/input/**/PlantVillageVQA.csv', recursive=True)
assert cands, 'add the PlantVillageVQA dataset as a Notebook input'
cfg['data']['data_root'] = os.path.dirname(cands[0])
cfg['data']['prepared_dir'] = '/kaggle/working/prepared'
cfg['cache']['feature_dir'] = '/kaggle/working/cache'
cfg['train']['out_dir'] = '/kaggle/working/runs/exp1'
yaml.safe_dump(cfg, open(cfg_path, 'w'))
print('data_root =', cfg['data']['data_root'])
# NOTE: on Kaggle the images dir may be lowercase 'images'; set data.images_dirname to match.

In [ ]:
!python -m data.prepare --config configs/default.yaml --subset-size 30000

In [ ]:
!python -m data.cache_features --config configs/default.yaml

In [ ]:
# sanity: overfit a tiny slice (loss should fall, EM rise)
!python train.py --config configs/default.yaml --overfit 40 --grad-accum 1 --epochs 40

In [ ]:
!python train.py --config configs/default.yaml

In [ ]:
!python eval.py --config configs/default.yaml --ckpt /kaggle/working/runs/exp1/best.pt --split test
!python eval.py --config configs/default.yaml --ckpt /kaggle/working/runs/exp1/best.pt --split test --blind
!python analyze_gates.py --ckpt /kaggle/working/runs/exp1/best.pt --out /kaggle/working/runs/exp1/gates.png